In [15]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split

podział bazujący na zbiorach DS1 i DS2 opisanych przez de Chazala i in. (2004), Automatic Classification of Heartbeats Using ECG Morphology and Heartbeat Interval Features, zmodyfikowany poprzez dodanie rekordów z uderzeniami stymulowanymi (klasa Q) oraz poprzez zmianę położenia rekordu 202 (z DS2 do DS1), ponieważ pochodzi on od tego samego pacjenta co 201.

In [16]:
DATA_PATH = '../src/clean_data.csv'
OUTPUT_PATH = '../src/'

RANDOM_STATE = 42

SEGMENT_LENGTH = 260

CLASSES = ['N', 'S', 'V', 'F', 'Q']

In [17]:
df = pd.read_csv(DATA_PATH)

df['record_id'] = df['record_id'].astype(int)

signal_columns = [f'x_{i}' for i in range(SEGMENT_LENGTH)]

assert all(col in df.columns for col in signal_columns)
assert df['class'].isin(CLASSES).all()

print(df.shape)
print(df['class'].value_counts())
display(df.head())

(109451, 266)
class
N    90593
Q     8040
V     7235
S     2781
F      802
Name: count, dtype: int64


,record_id,annotation_sample,type,class,anomaly,lead,x_0,x_1,x_2,x_3,...,x_250,x_251,x_252,x_253,x_254,x_255,x_256,x_257,x_258,x_259
0,100,370,N,N,0,MLII,0.011341,0.009784,0.007544,0.004454,...,0.011778,0.014272,0.016936,0.019570,0.021707,0.023014,0.023533,0.023590,0.023498,0.023357
1,100,662,N,N,0,MLII,0.004995,0.002567,0.000839,-0.000379,...,0.033399,0.032397,0.031036,0.029672,0.028491,0.027444,0.026337,0.025054,0.023703,0.022579
2,100,946,N,N,0,MLII,-0.014134,-0.018067,-0.020815,-0.021979,...,0.044939,0.042418,0.040438,0.039070,0.038409,0.038631,0.039813,0.041653,0.043418,0.044243
3,100,1231,N,N,0,MLII,0.009020,0.009817,0.009969,0.009328,...,0.021639,0.020224,0.019986,0.021034,0.022960,0.024955,0.026159,0.026098,0.024978,0.023616
4,100,1515,N,N,0,MLII,0.006367,0.007337,0.007091,0.005420,...,0.026973,0.027071,0.026602,0.026165,0.026145,0.026751,0.028042,0.029889,0.031937,0.033725


In [18]:
DS1_RECORDS = [
    101, 102, 104, 106, 108, 109, 112, 114,
    115, 116, 118, 119, 122, 124, 201, 202,
    203, 205, 207, 208, 209, 215, 220, 223, 230
]

DS2_RECORDS = [
    100, 103, 105, 107, 111, 113, 117, 121,
    123, 200, 210, 212, 213, 214, 217, 219,
    221, 222, 228, 231, 232, 233, 234
]

record_ids = sorted(df['record_id'].unique())

patient_groups = {
    rid: 201 if rid == 202 else rid
    for rid in record_ids
}

print('DS1:', len(DS1_RECORDS))
print('DS2:', len(DS2_RECORDS))
print('Wszystkie rekordy:', len(record_ids))

DS1: 25
DS2: 23
Wszystkie rekordy: 48


In [19]:
ds1_patients = {patient_groups[r] for r in DS1_RECORDS}
ds2_patients = {patient_groups[r] for r in DS2_RECORDS}

assert ds1_patients.isdisjoint(ds2_patients)

print('Podział DS1/DS2 jest poprawny.')

Podział DS1/DS2 jest poprawny.


In [20]:
ds1_df = df[
    df['record_id'].isin(DS1_RECORDS)
].copy()

ds2_df = df[
    df['record_id'].isin(DS2_RECORDS)
].copy()

assert len(ds1_df) + len(ds2_df) == len(df)

print('DS1:', ds1_df.shape)
print('DS2:', ds2_df.shape)

DS1: (57549, 266)
DS2: (51902, 266)


In [21]:
class_distribution = pd.DataFrame({
    'DS1': ds1_df['class'].value_counts(),
    'DS2': ds2_df['class'].value_counts()
}).reindex(CLASSES, fill_value=0)

display(class_distribution)

class_distribution_pct = (
    class_distribution
    / class_distribution.sum(axis=0)
    * 100
).round(2)

display(class_distribution_pct)

,DS1,DS2
class,,
N,48168,42425
S,999,1782
V,3813,3422
F,415,387
Q,4154,3886


,DS1,DS2
class,,
N,83.70,81.74
S,1.74,3.43
V,6.63,6.59
F,0.72,0.75
Q,7.22,7.49


In [22]:
ds1_df.to_csv(
    OUTPUT_PATH + 'ds1.csv',
    index=False
)

ds2_df.to_csv(
    OUTPUT_PATH + 'ds2.csv',
    index=False
)

In [23]:
split_records = pd.DataFrame(
    [(rid, 'DS1') for rid in DS1_RECORDS]
    + [(rid, 'DS2') for rid in DS2_RECORDS],
    columns=['record_id', 'split']
)

split_records = split_records.sort_values('record_id')

split_records.to_csv(
    OUTPUT_PATH + 'split_records.csv',
    index=False
)

In [24]:
from sklearn.model_selection import StratifiedGroupKFold

cv = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE
)

groups = ds1_df['record_id'].replace({202: 201})

fold_results = []

for fold, (train_idx, val_idx) in enumerate(
    cv.split(ds1_df, ds1_df['class'], groups=groups),
    start=1
):
    train_fold = ds1_df.iloc[train_idx]
    val_fold = ds1_df.iloc[val_idx]

    assert set(groups.iloc[train_idx]).isdisjoint(
        set(groups.iloc[val_idx])
    )

    counts = (
        val_fold['class']
        .value_counts()
        .reindex(CLASSES, fill_value=0)
    )

    fold_results.append({
        'fold': fold,
        'records': val_fold['record_id'].nunique(),
        'samples': len(val_fold),
        **counts.to_dict()
    })

fold_summary = pd.DataFrame(fold_results)

display(fold_summary)

,fold,records,samples,N,S,V,F,Q
0,1,5,10204,9351,186,662,3,2
1,2,5,11904,9151,205,485,1,2062
2,3,5,11725,10347,389,982,3,4
3,4,5,11540,8724,107,595,30,2084
4,5,5,12176,10595,112,1089,378,2


In [25]:
for fold, (train_idx, val_idx) in enumerate(
    cv.split(ds1_df, ds1_df['class'], groups=groups),
    start=1
):
    train_counts = (
        ds1_df.iloc[train_idx]['class']
        .value_counts()
        .reindex(CLASSES, fill_value=0)
    )

    print(f'Fold {fold}: {train_counts.to_dict()}')

    assert (train_counts > 0).all()

Fold 1: {'N': 38817, 'S': 813, 'V': 3151, 'F': 412, 'Q': 4152}
Fold 2: {'N': 39017, 'S': 794, 'V': 3328, 'F': 414, 'Q': 2092}
Fold 3: {'N': 37821, 'S': 610, 'V': 2831, 'F': 412, 'Q': 4150}
Fold 4: {'N': 39444, 'S': 892, 'V': 3218, 'F': 385, 'Q': 2070}
Fold 5: {'N': 37573, 'S': 887, 'V': 2724, 'F': 37, 'Q': 4152}


In [26]:
record_distribution = pd.crosstab(
    ds1_df['record_id'],
    ds1_df['class']
).reindex(columns=CLASSES, fill_value=0)

display(record_distribution)

class,N,S,V,F,Q
record_id,,,,,
101,1859,3,0,0,2
102,99,0,4,0,2084
104,163,0,2,0,2062
106,1507,0,520,0,0
108,1739,4,17,2,0
109,2490,0,38,2,0
112,2535,2,0,0,0
114,1820,12,43,4,0
115,1952,0,0,0,0
